In [ ]:
#!pip install transformers datasets sentencepiece accelerate sacrebleu -q
#!pip install --no-cache-dir transformers==4.40.2 accelerate==0.29.3 datasets sentencepiece -q


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

df = pd.read_csv("//content/BHILI_Hindi_NEW.csv")

df = df[["Hindi", "Bhili"]].dropna()
df = df.rename(columns={"Hindi": "hindi", "Bhili": "bhili"})

# First split: 90% train, 10% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

# Split temp into 5% val and 5% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)


Train: 9000
Validation: 500
Test: 500


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer.src_lang = "hin_Deva"
tokenizer.tgt_lang = "hin_Deva"

model = model.to("cuda")

print("Model loaded successfully")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

Model loaded successfully


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer.src_lang = "hin_Deva"
tokenizer.tgt_lang = "hin_Deva"

model = model.to("cuda")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [7]:
max_length = 128

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["hindi"],
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["bhili"],
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
#!pip install --no-cache-dir transformers==4.40.2 accelerate==0.29.3 datasets sentencepiece -q
#!pip uninstall -y transformers accelerate peft
#!pip install --no-cache-dir transformers==4.40.2 accelerate==0.29.3 datasets sentencepiece


In [1]:
import transformers
print(transformers.__version__)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

4.40.2


In [11]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./hindi-bhili-production",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=100,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss
1,0.405900,0.356645
2,0.337200,0.321732
3,0.300300,0.313798


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation 

TrainOutput(global_step=6750, training_loss=0.7230847738760489, metrics={'train_runtime': 3732.761, 'train_samples_per_second': 7.233, 'train_steps_per_second': 1.808, 'total_flos': 7313978032128000.0, 'train_loss': 0.7230847738760489, 'epoch': 3.0})

In [12]:
trainer.save_model("./final-hindi-bhili-model")
tokenizer.save_pretrained("./final-hindi-bhili-model")

print("Best model weights saved successfully.")


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200}


Best model weights saved successfully.


In [13]:
!pip install sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00


In [14]:
import sacrebleu
from tqdm import tqdm

model.eval()

predictions = []
references = []
sources = []

for example in tqdm(test_dataset):
    input_text = example["hindi"]

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128)

    decoded_pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

    predictions.append(decoded_pred)
    references.append(example["bhili"])
    sources.append(input_text)

bleu = sacrebleu.corpus_bleu(predictions, [references])
chrf = sacrebleu.corpus_chrf(predictions, [references])

print("Average BLEU Score (100):", round(bleu.score, 2))
print("Average chrF Score (100):", round(chrf.score, 2))


100%|██████████| 500/500 [07:35<00:00,  1.10it/s]


Average BLEU Score (100): 26.22
Average chrF Score (100): 56.76


In [15]:
import pandas as pd

results_df = pd.DataFrame({
    "Hindi": sources,
    "Actual_Bhili": references,
    "Predicted_Bhili": predictions,
    "BLEU_score": [round(bleu.score, 2)] * len(sources),
    "chrF_score": [round(chrf.score, 2)] * len(sources)
})

results_df.to_csv("final_report.csv", index=False)

print("Final report CSV saved as final_report.csv")


Final report CSV saved as final_report.csv


In [16]:
train_sample_200 = train_df.sample(200, random_state=42)
train_sample_200.to_csv("train_200_sample_report.csv", index=False)

print("200 training rows saved as train_200_sample_report.csv")


200 training rows saved as train_200_sample_report.csv


In [21]:
report=pd.read_csv('final_report.csv')
print(report.head(5).to_string())

                                                                                                                                                      Hindi                                                                                                                                  Actual_Bhili                                                                                                                           Predicted_Bhili  BLEU_score  chrF_score
0                                                                                 मेरे पास आकर, वह दरवाजा खोला, और बोला: "यह झूला तुम्हारे बच्चे के लिए है।                                                                       मारी कन आवीन त्यो दरवाजो खोलयो अन बोल्यो कि आ झूलों तमारा सौरा हारू से।                                                                     मारी कन आवीन त्यो दरवाजा खोल्यो अने केदुं यो झूला तमारी सोरा हारू से।       26.22       56.76
1                                                           हमारे खेत के किसान अ